In [2]:
# Run only once

!pip install pandas numpy pyarrow fastparquet scikit-learn \
xgboost lightgbm shap matplotlib tqdm joblib scipy \
imbalanced-learn jupyterlab notebook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 114.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import glob
import time
import gc
import shutil
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# =========================================================
# CESNET-QUIC22 STREAMING CONFIGURATION
# =========================================================
# This configuration is optimized for:
#
# - Google Colab
# - Low Google Drive storage
# - Week-by-week processing
# - Incremental feature extraction
# - Large PCAP datasets
#
# Workflow:
#
# Upload ONE zip
# → Extract
# → Process
# → Save parquet features
# → Delete raw files
# → Upload next zip
# =========================================================

# ---------------------------------------------------------
# BASE DIRECTORY
# ---------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING"
)

# ---------------------------------------------------------
# INPUT ZIP FOLDER
# ---------------------------------------------------------
# Upload ONE week zip here at a time
#
# Example:
# W-2022-44.zip
# ---------------------------------------------------------

INCOMING_ZIP_DIR = BASE_DIR / "incoming_zip"

# ---------------------------------------------------------
# TEMP EXTRACTION DIRECTORY
# ---------------------------------------------------------
# Raw PCAPs are extracted here temporarily
# and deleted after processing
# ---------------------------------------------------------

TEMP_EXTRACT_DIR = BASE_DIR / "temp_extract"

# ---------------------------------------------------------
# FINAL FEATURE STORAGE
# ---------------------------------------------------------
# Only lightweight parquet feature files stay permanently
# ---------------------------------------------------------

FEATURE_DIR = BASE_DIR / "weekly_features"

# ---------------------------------------------------------
# FINAL MERGED DATASET
# ---------------------------------------------------------

MASTER_DATASET_DIR = BASE_DIR / "master_dataset"

# ---------------------------------------------------------
# MODEL STORAGE
# ---------------------------------------------------------

MODEL_DIR = BASE_DIR / "models"

# ---------------------------------------------------------
# CREATE DIRECTORIES
# ---------------------------------------------------------

for d in [
    INCOMING_ZIP_DIR,
    TEMP_EXTRACT_DIR,
    FEATURE_DIR,
    MASTER_DATASET_DIR,
    MODEL_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

# =========================================================
# PARQUET CONFIGURATION
# =========================================================

PARQUET_ENGINE = "pyarrow"

PARQUET_COMPRESSION = "snappy"

# =========================================================
# MEMORY OPTIMIZATION
# =========================================================

CHUNK_SIZE = 200000

# Optional:
# Limit PCAPs during testing

MAX_PCAPS_PER_WEEK = None

# =========================================================
# AUTO-DETECT ZIP FILES
# =========================================================
# Example:
# incoming_zip/
# ├── W-2022-44.zip
# ├── W-2022-45.zip
# =========================================================

ZIP_FILES = sorted(
    INCOMING_ZIP_DIR.glob("*.zip")
)

print("=" * 60)
print("DETECTED ZIP FILES")
print("=" * 60)

for z in ZIP_FILES:
    print(z.name)

# =========================================================
# EXTRACT WEEK ZIP
# =========================================================

def extract_week_zip(zip_path):

    week_name = zip_path.stem

    output_dir = TEMP_EXTRACT_DIR / week_name

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"\nExtracting {week_name}...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(output_dir)

    return week_name, output_dir

# =========================================================
# GET ALL PCAP FILES
# =========================================================

def get_week_pcaps(extract_dir):

    pcap_files = sorted(
        glob.glob(
            str(extract_dir / "**/*.pcap"),
            recursive=True
        )
    )

    pcapng_files = sorted(
        glob.glob(
            str(extract_dir / "**/*.pcapng"),
            recursive=True
        )
    )

    all_pcaps = pcap_files + pcapng_files

    if MAX_PCAPS_PER_WEEK:
        all_pcaps = all_pcaps[:MAX_PCAPS_PER_WEEK]

    return all_pcaps

# =========================================================
# OUTPUT PATHS
# =========================================================

def get_output_paths(week_name):

    return {
        "feature_parquet":
            FEATURE_DIR / f"{week_name}.parquet",

        "master_parquet":
            MASTER_DATASET_DIR / "all_weeks.parquet"
    }

# =========================================================
# CLEANUP TEMP FILES
# =========================================================
# Deletes:
# - extracted raw pcaps
# - uploaded zip
# =========================================================

def cleanup_week(zip_path, extract_dir):

    print("\nCleaning temporary files...")

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    if zip_path.exists():
        os.remove(zip_path)

    gc.collect()

    print("Cleanup complete.")

# =========================================================
# DISPLAY CURRENT STORAGE STATUS
# =========================================================

def show_storage():

    feature_files = sorted(
        FEATURE_DIR.glob("*.parquet")
    )

    print("\nSaved Weekly Feature Files:")

    for f in feature_files:

        size_mb = f.stat().st_size / (1024 * 1024)

        print(
            f"{f.name} -> "
            f"{size_mb:.2f} MB"
        )

# =========================================================
# READY FOR PROCESSING
# =========================================================

print("\n")
print("=" * 60)
print("STREAMING PIPELINE READY")
print("=" * 60)

print(f"ZIP INPUT DIR       : {INCOMING_ZIP_DIR}")
print(f"TEMP EXTRACT DIR    : {TEMP_EXTRACT_DIR}")
print(f"FEATURE STORAGE DIR : {FEATURE_DIR}")
print(f"MASTER DATASET DIR  : {MASTER_DATASET_DIR}")
print(f"MODEL DIR           : {MODEL_DIR}")
print("=" * 60)

DETECTED ZIP FILES
W-2022-47.zip


STREAMING PIPELINE READY
ZIP INPUT DIR       : /content/drive/MyDrive/CESNET_STREAMING/incoming_zip
TEMP EXTRACT DIR    : /content/drive/MyDrive/CESNET_STREAMING/temp_extract
FEATURE STORAGE DIR : /content/drive/MyDrive/CESNET_STREAMING/weekly_features
MASTER DATASET DIR  : /content/drive/MyDrive/CESNET_STREAMING/master_dataset
MODEL DIR           : /content/drive/MyDrive/CESNET_STREAMING/models


In [ ]:
import zipfile

# =====================================================
# EXTRACT ALL DETECTED ZIP FILES
# =====================================================

for zip_file in ZIP_FILES:

    print("\n" + "=" * 60)
    print(f"PROCESSING: {zip_file.name}")
    print("=" * 60)

    # -------------------------------------------------
    # Week name
    # -------------------------------------------------

    week_name = zip_file.stem

    # -------------------------------------------------
    # Extraction path
    # -------------------------------------------------

    extract_dir = TEMP_EXTRACT_DIR / week_name

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # -------------------------------------------------
    # Extract ZIP
    # -------------------------------------------------

    print(f"\nExtracting {week_name}...")

    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)

    print("Extraction completed.")

    # -------------------------------------------------
    # Find csv.gz files
    # -------------------------------------------------

    csv_files = sorted(
        glob.glob(
            str(extract_dir / "**/*.csv.gz"),
            recursive=True
        )
    )

    print(f"\nFound {len(csv_files)} csv.gz files")

    # -------------------------------------------------
    # Show first few files
    # -------------------------------------------------

    for f in csv_files[:5]:
        print(Path(f).name)


PROCESSING: W-2022-47.zip

Extracting W-2022-47...
Extraction completed.

Found 7 csv.gz files
flows-20221121.csv.gz
flows-20221122.csv.gz
flows-20221123.csv.gz
flows-20221124.csv.gz
flows-20221125.csv.gz


In [ ]:
# =========================================================
# HYBRID LABEL ENGINEERING
# =========================================================

SNI_MAP = {

    # =====================================================
    # GOOGLE ECOSYSTEM
    # =====================================================

    "google": "google",
    "googleapis": "google",
    "gstatic": "google",
    "googleusercontent": "google",
    "googlevideo": "youtube",
    "ytimg": "youtube",
    "youtube": "youtube",
    "youtu": "youtube",
    "ggpht": "google_photos",
    "gmail": "gmail",
    "android": "android",
    "chromecast": "google",
    "widevine": "google",
    "gvt1": "google",
    "gvt2": "google",
    "1e100": "google",

    # =====================================================
    # META / FACEBOOK
    # =====================================================

    "facebook": "facebook",
    "fbcdn": "facebook",
    "fbsbx": "facebook",
    "instagram": "instagram",
    "cdninstagram": "instagram",
    "whatsapp": "whatsapp",
    "whatsapp.net": "whatsapp",
    "messenger": "messenger",

    # =====================================================
    # NETFLIX
    # =====================================================

    "netflix": "netflix",
    "nflxvideo": "netflix",
    "nflximg": "netflix",
    "nflxext": "netflix",
    "nflxso": "netflix",

    # =====================================================
    # AMAZON
    # =====================================================

    "amazon": "amazon",
    "amazonaws": "aws",
    "aws": "aws",
    "primevideo": "primevideo",
    "cloudfront": "aws",
    "alexa": "amazon",

    # =====================================================
    # MICROSOFT
    # =====================================================

    "microsoft": "microsoft",
    "office": "office365",
    "office365": "office365",
    "live": "microsoft",
    "outlook": "outlook",
    "skype": "skype",
    "teams": "teams",
    "xbox": "xbox",
    "azure": "azure",
    "bing": "bing",

    # =====================================================
    # APPLE
    # =====================================================

    "apple": "apple",
    "icloud": "icloud",
    "itunes": "itunes",
    "mzstatic": "apple",
    "apple-dns": "apple",
    "facetime": "facetime",

    # =====================================================
    # STREAMING
    # =====================================================

    "spotify": "spotify",
    "deezer": "deezer",
    "soundcloud": "soundcloud",
    "twitch": "twitch",
    "disneyplus": "disneyplus",
    "hulu": "hulu",
    "hbomax": "hbomax",
    "hotstar": "hotstar",
    "jiocinema": "jiocinema",
    "zee5": "zee5",
    "sonyliv": "sonyliv",

    # =====================================================
    # SOCIAL / CHAT
    # =====================================================

    "discord": "discord",
    "discordapp": "discord",
    "telegram": "telegram",
    "t.me": "telegram",
    "snapchat": "snapchat",
    "reddit": "reddit",
    "twitter": "twitter",
    "x.com": "twitter",
    "linkedin": "linkedin",
    "quora": "quora",
    "pinterest": "pinterest",

    # =====================================================
    # VIDEO CALL
    # =====================================================

    "zoom": "zoom",
    "meet": "google_meet",
    "webex": "webex",
    "slack": "slack",

    # =====================================================
    # CDN / CLOUD
    # =====================================================

    "cloudflare": "cloudflare",
    "akamai": "akamai",
    "fastly": "fastly",
    "cdn": "cdn",
    "edgekey": "akamai",
    "edgesuite": "akamai",

    # =====================================================
    # GAMING
    # =====================================================

    "epicgames": "epicgames",
    "steam": "steam",
    "steampowered": "steam",
    "riotgames": "riotgames",
    "valorant": "valorant",
    "minecraft": "minecraft",
    "roblox": "roblox",
    "ea": "ea_games",
    "ubisoft": "ubisoft",

    # =====================================================
    # AI / DEV
    # =====================================================

    "openai": "openai",
    "chatgpt": "openai",
    "github": "github",
    "githubusercontent": "github",
    "gitlab": "gitlab",
    "stackoverflow": "stackoverflow",
    "docker": "docker",
    "huggingface": "huggingface",

    # =====================================================
    # E-COMMERCE
    # =====================================================

    "flipkart": "flipkart",
    "myntra": "myntra",
    "ajio": "ajio",
    "ebay": "ebay",
    "walmart": "walmart",
    "etsy": "etsy",

    # =====================================================
    # AD NETWORKS / TRACKING
    # =====================================================

    "doubleclick": "ads",
    "googlesyndication": "ads",
    "adservice": "ads",
    "taboola": "ads",
    "outbrain": "ads",

    # =====================================================
    # EDUCATION
    # =====================================================

    "coursera": "coursera",
    "udemy": "udemy",
    "edx": "edx",
    "khanacademy": "khanacademy",

    # =====================================================
    # SECURITY
    # =====================================================

    "cloudflare-dns": "dns",
    "dns.google": "dns",
    "quad9": "dns",
    "opendns": "dns",

}



# =========================================================
# LABEL FROM SNI
# =========================================================

def get_label_from_sni(sni):

    if pd.isna(sni):
        return "unknown"

    sni = str(sni).lower().strip()

    for pattern, label in SNI_MAP.items():

        if pattern in sni:
            return label

    return "unknown"

# =========================================================
# APPLY LABELS
# =========================================================

def apply_hybrid_labels(df):
    possible_cols = [ "QUIC_SNI", "SNI", "tls_sni", "server_name" ]


    sni_col = None

    for col in possible_cols:

        if col in df.columns:
            sni_col = col
            break

    if sni_col:

        df["traffic_label"] = (
            df[sni_col]
            .apply(get_label_from_sni)
        )

    else:

        df["traffic_label"] = "unknown"

    return df

# =========================================================
# FEATURE EXTRACTION
# =========================================================

def extract_hybrid_features(df):

    df = apply_hybrid_labels(df)

    features = pd.DataFrame()

    # -----------------------------------------------------
    # LABEL
    # -----------------------------------------------------

    features["label"] = df["traffic_label"]

    # -----------------------------------------------------
    # FLOW FEATURES
    # -----------------------------------------------------

    packet_col = (
        "PACKETS"
        if "PACKETS" in df.columns
        else "packets"
    )

    byte_col = (
        "BYTES"
        if "BYTES" in df.columns
        else "bytes"
    )

    duration_col = (
        "DURATION"
        if "DURATION" in df.columns
        else "duration"
    )

    # -----------------------------------------------------
    # SAFE EXTRACTION
    # -----------------------------------------------------

    features["packet_count"] = (
        df[packet_col]
        .fillna(0)
        .astype("int32")
    )

    features["total_bytes"] = (
        df[byte_col]
        .fillna(0)
        .astype("int32")
    )

    features["duration_s"] = (
        df[duration_col]
        .fillna(0)
        .astype("float32")
    )

    # -----------------------------------------------------
    # DERIVED FEATURES
    # -----------------------------------------------------

    features["bytes_per_sec"] = np.where(
        features["duration_s"] > 0,

        features["total_bytes"] /
        features["duration_s"],

        0
    ).astype("float32")

    features["packets_per_sec"] = np.where(
        features["duration_s"] > 0,

        features["packet_count"] /
        features["duration_s"],

        0
    ).astype("float32")

    features["mean_payload_len"] = np.where(
        features["packet_count"] > 0,

        features["total_bytes"] /
        features["packet_count"],

        0
    ).astype("float32")

    # -----------------------------------------------------
    # REVERSE FLOW FEATURES
    # -----------------------------------------------------

    if "PACKETS_REV" in df.columns:

        rev_packets = (
            df["PACKETS_REV"]
            .fillna(0)
            .astype("int32")
        )

        features["upload_ratio"] = (
            rev_packets /
            np.maximum(
                features["packet_count"],
                1
            )
        ).astype("float32")

    else:

        features["upload_ratio"] = 0.0

    # -----------------------------------------------------
    # FLOW FILTERING
    # -----------------------------------------------------

    features = features[
        features["packet_count"] > 2
    ]

    features = features[
        features["total_bytes"] > 100
    ]

    return features

In [ ]:
# =========================================================
# STREAM LARGE CSV.GZ FILES
# =========================================================

def stream_csv_gz(csv_file):

    for chunk in pd.read_csv(
        csv_file,
        compression='gzip',
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        yield chunk

In [ ]:
# =========================================================
# PROCESS EXTRACTED WEEK
# =========================================================

def process_current_week(
    week_name,
    csv_files
):

    print("\n" + "="*60)
    print(f"PROCESSING WEEK: {week_name}")
    print("="*60)

    weekly_feature_chunks = []

    total_rows = 0

    # -----------------------------------------------------
    # LOOP THROUGH DAILY CSV FILES
    # -----------------------------------------------------

    for file_idx, csv_file in enumerate(csv_files):

        print(
            f"\n[{file_idx+1}/{len(csv_files)}] "
            f"{Path(csv_file).name}"
        )

        # -------------------------------------------------
        # STREAM CHUNKS
        # -------------------------------------------------

        for chunk_idx, chunk in enumerate(
            stream_csv_gz(csv_file)
        ):

            print(
                f"  Chunk {chunk_idx+1} "
                f"Shape: {chunk.shape}"
            )

            # =============================================
            # FEATURE EXTRACTION
            # =============================================

            features = extract_hybrid_features(chunk)

            # =============================================
            # REMOVE UNKNOWN LABELS
            # =============================================

            features = features[
                features["label"] != "unknown"
            ]

            total_rows += len(features)

            # =============================================
            # STORE FEATURES
            # =============================================

            weekly_feature_chunks.append(features)

            # =============================================
            # MEMORY CLEANUP
            # =============================================

            del chunk
            del features

            gc.collect()

    # =====================================================
    # MERGE WEEK
    # =====================================================

    print("\nMerging weekly feature chunks...")

    week_df = pd.concat(
        weekly_feature_chunks,
        ignore_index=True
    )

    # =====================================================
    # SAVE WEEKLY PARQUET
    # =====================================================

    output_path = (
        FEATURE_DIR /
        f"{week_name}_features.parquet"
    )

    week_df.to_parquet(
        output_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

    print("\n" + "="*60)
    print("WEEK PROCESSING COMPLETED")
    print("="*60)

    print(f"Total labelled flows: {len(week_df)}")

    print(f"\nSaved parquet:\n{output_path}")

    # =====================================================
    # CLEAN MEMORY
    # =====================================================

    del week_df
    del weekly_feature_chunks

    gc.collect()

    return output_path

In [ ]:
# =========================================================
# GENERATE WEEKLY PARQUET
# =========================================================

week_name = "W-2022-47"

output_path = process_current_week(
    week_name,
    csv_files
)

print("\nGenerated parquet:")
print(output_path)


PROCESSING WEEK: W-2022-47

[1/7] flows-20221121.csv.gz
  Chunk 1 Shape: (200000, 30)
  Chunk 2 Shape: (200000, 30)
  Chunk 3 Shape: (200000, 30)
  Chunk 4 Shape: (200000, 30)
  Chunk 5 Shape: (200000, 30)
  Chunk 6 Shape: (200000, 30)
  Chunk 7 Shape: (200000, 30)
  Chunk 8 Shape: (200000, 30)
  Chunk 9 Shape: (200000, 30)
  Chunk 10 Shape: (200000, 30)
  Chunk 11 Shape: (200000, 30)
  Chunk 12 Shape: (200000, 30)
  Chunk 13 Shape: (200000, 30)
  Chunk 14 Shape: (200000, 30)
  Chunk 15 Shape: (200000, 30)
  Chunk 16 Shape: (200000, 30)
  Chunk 17 Shape: (200000, 30)
  Chunk 18 Shape: (200000, 30)
  Chunk 19 Shape: (200000, 30)
  Chunk 20 Shape: (200000, 30)
  Chunk 21 Shape: (200000, 30)
  Chunk 22 Shape: (200000, 30)
  Chunk 23 Shape: (200000, 30)
  Chunk 24 Shape: (200000, 30)
  Chunk 25 Shape: (200000, 30)
  Chunk 26 Shape: (200000, 30)
  Chunk 27 Shape: (200000, 30)
  Chunk 28 Shape: (200000, 30)
  Chunk 29 Shape: (200000, 30)
  Chunk 30 Shape: (200000, 30)
  Chunk 31 Shape: (200

In [6]:
import glob
from pathlib import Path

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING/weekly_features"
)

all_parquets = glob.glob(
    str(PROCESSED_DIR / "*_features.parquet")
)

print("Found", len(all_parquets), "files")

for f in all_parquets:
    print(f)

Found 4 files
/content/drive/MyDrive/CESNET_STREAMING/weekly_features/W-2022-45_features.parquet
/content/drive/MyDrive/CESNET_STREAMING/weekly_features/W-2022-44_features.parquet
/content/drive/MyDrive/CESNET_STREAMING/weekly_features/W-2022-46_features.parquet
/content/drive/MyDrive/CESNET_STREAMING/weekly_features/W-2022-47_features.parquet


In [3]:
import os

for p in all_parquets:
    print(
        Path(p).name,
        round(os.path.getsize(p)/(1024**3), 2),
        "GB"
    )

W-2022-45_features.parquet 0.73 GB
W-2022-44_features.parquet 0.55 GB
W-2022-46_features.parquet 0.58 GB
W-2022-47_features.parquet 0.76 GB


In [4]:
import pandas as pd

df = pd.read_parquet(
    "/content/drive/MyDrive/CESNET_STREAMING/weekly_features/W-2022-47_features.parquet"
)

print(df.shape)
print(df["label"].value_counts())

(39288106, 8)
label
google           24527591
facebook          4392436
instagram         1783949
spotify           1649817
youtube           1121500
discord           1009965
snapchat           881044
cdn                875902
ads                581492
icloud             473850
ea_games           450777
cloudflare         425437
aws                304453
office365          270073
messenger          208547
google_photos      165630
whatsapp            86137
microsoft           51188
twitter              6567
quora                5364
telegram             5108
ebay                 2555
pinterest            2179
epicgames            1918
hbomax               1621
zoom                 1065
twitch                487
google_meet           215
akamai                204
steam                 183
android               150
outlook               112
bing                   92
minecraft              65
apple                  62
docker                 58
reddit                 54
openai            

In [1]:

import pandas as pd
import numpy as np
import glob
import gc
import joblib

from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print(f"\n{'='*50}")
print("Preparing Dataset Incrementally")
print(f"{'='*50}")

# =====================================================
# PATHS
# =====================================================

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING/weekly_features"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING/train_test_data"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)

# =====================================================
# FIND PARQUETS
# =====================================================

all_parquets = sorted(
    glob.glob(
        str(PROCESSED_DIR / "*_features.parquet")
    )
)

print("Weeks Found:")

for p in all_parquets:
    print(Path(p).name)

# =====================================================
# BUILD GLOBAL LABEL ENCODER
# =====================================================

all_labels = set()

for parquet_file in all_parquets:

    labels = pd.read_parquet(
        parquet_file,
        columns=["label"]
    )

    all_labels.update(
        labels["label"].unique()
    )

    del labels
    gc.collect()

le = LabelEncoder()

le.fit(
    list(all_labels)
)

joblib.dump(
    le,
    OUTPUT_DIR / "label_encoder.pkl"
)

print(
    f"\nTotal Classes: "
    f"{len(le.classes_)}"
)

# =====================================================
# PROCESS WEEK BY WEEK
# =====================================================

for parquet_file in all_parquets:

    week_name = Path(
        parquet_file
    ).stem

    print(
        f"\nProcessing "
        f"{week_name}"
    )

    df = pd.read_parquet(
        parquet_file
    )

    df = (
        df
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    X = df.drop(
        columns=["label"]
    )

    y = le.transform(
        df["label"]
    )

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=42
        )
    )

    train_df = X_train.copy()
    train_df["label"] = y_train

    test_df = X_test.copy()
    test_df["label"] = y_test

    train_df.to_parquet(
        OUTPUT_DIR /
        f"{week_name}_train.parquet",
        index=False
    )

    test_df.to_parquet(
        OUTPUT_DIR /
        f"{week_name}_test.parquet",
        index=False
    )

    print(
        f"Train Rows: "
        f"{len(train_df):,}"
    )

    print(
        f"Test Rows: "
        f"{len(test_df):,}"
    )

    del df
    del X
    del y
    del train_df
    del test_df

    gc.collect()

print(
    "\nAll Weeks Processed!"
)




Preparing Dataset Incrementally
Weeks Found:
W-2022-44_features.parquet
W-2022-45_features.parquet
W-2022-46_features.parquet
W-2022-47_features.parquet

Total Classes: 66

Processing W-2022-44_features
Train Rows: 22,831,443
Test Rows: 5,707,861

Processing W-2022-45_features
Train Rows: 30,169,350
Test Rows: 7,542,338

Processing W-2022-46_features
Train Rows: 23,897,532
Test Rows: 5,974,383

Processing W-2022-47_features
Train Rows: 31,430,484
Test Rows: 7,857,622

All Weeks Processed!


In [2]:
import pandas as pd
import numpy as np
import glob
import gc
import joblib
import time
import xgboost as xgb
import pyarrow.parquet as pq

from pathlib import Path

# =====================================================
# PATHS
# =====================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING/train_test_data"
)

MODEL_PATH = (
    DATA_DIR /
    "cesnet_xgboost_model.json"
)

ENCODER_PATH = (
    DATA_DIR /
    "label_encoder.pkl"
)

CHECKPOINT_PATH = (
    DATA_DIR /
    "training_checkpoint.pkl"
)

# =====================================================
# LOAD LABEL ENCODER
# =====================================================

le = joblib.load(
    ENCODER_PATH
)

num_classes = len(
    le.classes_
)

print(
    f"\nTotal Classes: {num_classes}"
)

# =====================================================
# FIND TRAIN FILES
# =====================================================

train_files = sorted(
    glob.glob(
        str(DATA_DIR / "*_train.parquet")
    )
)

print("\nTraining Files:")

for f in train_files:
    print(Path(f).name)

# =====================================================
# XGBOOST PARAMETERS
# =====================================================

params = {
    "objective": "multi:softprob",
    "num_class": num_classes,
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "seed": 42,
    "verbosity": 1
}

# =====================================================
# LOAD CHECKPOINT
# =====================================================

if CHECKPOINT_PATH.exists():

    checkpoint = joblib.load(
        CHECKPOINT_PATH
    )

    start_week = checkpoint[
        "week_idx"
    ]

    start_batch = (
        checkpoint[
            "batch_idx"
        ] + 1
    )

    first_batch = False

    print(
        f"\nResuming Training"
    )

    print(
        f"Week Index : {start_week}"
    )

    print(
        f"Batch Index: {start_batch}"
    )

else:

    start_week = 0
    start_batch = 0

    first_batch = True

    print(
        "\nStarting Fresh Training"
    )

# =====================================================
# TRAINING LOOP
# =====================================================

for week_idx, train_file in enumerate(
    train_files
):

    # -----------------------------------------
    # SKIP COMPLETED WEEKS
    # -----------------------------------------

    if week_idx < start_week:
        continue

    print("\n" + "=" * 60)

    print(
        f"PROCESSING "
        f"{Path(train_file).name}"
    )

    print("=" * 60)

    parquet_file = pq.ParquetFile(
        train_file
    )

    # -----------------------------------------
    # BATCH LOOP
    # -----------------------------------------

    for batch_idx, batch in enumerate(

        parquet_file.iter_batches(
            batch_size=1500_000
        )

    ):

        # -------------------------------------
        # SKIP COMPLETED BATCHES
        # -------------------------------------

        if (
            week_idx == start_week
            and batch_idx < start_batch
        ):
            continue

        t0 = time.time()

        # -------------------------------------
        # CONVERT TO DATAFRAME
        # -------------------------------------

        df = batch.to_pandas()

        X_train = (
            df
            .drop(
                columns=["label"]
            )
            .astype(np.float32)
        )

        y_train = (
            df["label"]
            .astype(np.int32)
            .values
        )

        # -------------------------------------
        # CLASS WEIGHTS
        # -------------------------------------

        class_counts = np.bincount(
            y_train,
            minlength=num_classes
        )

        class_counts[
            class_counts == 0
        ] = 1

        class_weights = np.sqrt(
            len(y_train)
            /
            (
                num_classes
                * class_counts
            )
        )

        sample_weights = (
            class_weights[
                y_train
            ]
        )

        # -------------------------------------
        # DMatrix
        # -------------------------------------

        dtrain = xgb.DMatrix(
            X_train,
            label=y_train,
            weight=sample_weights
        )

        # -------------------------------------
        # TRAIN MODEL
        # -------------------------------------

        if first_batch:

            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=5
            )

            first_batch = False

        else:

            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=5,
                xgb_model=str(MODEL_PATH)
            )

        # -------------------------------------
        # SAVE MODEL
        # -------------------------------------

        booster.save_model(
            str(MODEL_PATH)
        )

        # -------------------------------------
        # SAVE CHECKPOINT
        # -------------------------------------

        joblib.dump(
            {
                "week_idx": week_idx,
                "batch_idx": batch_idx,
                "timestamp": time.time()
            },
            CHECKPOINT_PATH
        )

        print(
            f"Week {week_idx+1} | "
            f"Batch {batch_idx+1} | "
            f"Rows {len(df):,} | "
            f"{time.time()-t0:.1f}s"
        )

        # -------------------------------------
        # CLEANUP
        # -------------------------------------

        del df
        del X_train
        del y_train
        del sample_weights
        del dtrain

        gc.collect()

# =====================================================
# TRAINING COMPLETE
# =====================================================

if CHECKPOINT_PATH.exists():

    CHECKPOINT_PATH.unlink()

joblib.dump(
    le,
    ENCODER_PATH
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    f"\nFinal Model Saved:\n"
    f"{MODEL_PATH}"
)

print(
    "\nCheckpoint Removed."
)


Total Classes: 66

Training Files:
W-2022-44_features_train.parquet
W-2022-45_features_train.parquet
W-2022-46_features_train.parquet
W-2022-47_features_train.parquet

Resuming Training
Week Index : 3
Batch Index: 19

PROCESSING W-2022-47_features_train.parquet
Week 4 | Batch 20 | Rows 1,500,000 | 693.0s
Week 4 | Batch 21 | Rows 1,430,484 | 693.7s

TRAINING COMPLETE

Final Model Saved:
/content/drive/MyDrive/CESNET_STREAMING/train_test_data/cesnet_xgboost_model.json

Checkpoint Removed.


In [1]:

import pandas as pd
import numpy as np
import glob
import joblib
import xgboost as xgb

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

# =====================================================
# PATHS
# =====================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/CESNET_STREAMING/train_test_data"
)

MODEL_PATH = (
    DATA_DIR /
    "cesnet_xgboost_model.json"
)

ENCODER_PATH = (
    DATA_DIR /
    "label_encoder.pkl"
)

# =====================================================
# LOAD MODEL
# =====================================================

booster = xgb.Booster()
booster.load_model(str(MODEL_PATH))

le = joblib.load(
    ENCODER_PATH
)

num_classes = len(
    le.classes_
)

# =====================================================
# TEST FILES
# =====================================================

test_files = sorted(
    glob.glob(
        str(DATA_DIR / "*_test.parquet")
    )
)

# =====================================================
# METRIC ACCUMULATORS
# =====================================================

all_y_true = []
all_y_pred = []

for test_file in test_files:

    print(
        f"\nEvaluating "
        f"{Path(test_file).name}"
    )

    df = pd.read_parquet(
        test_file
    )

    X_test = (
        df
        .drop(columns=["label"])
        .astype(np.float32)
    )

    y_test = (
        df["label"]
        .astype(np.int32)
        .values
    )

    dtest = xgb.DMatrix(
        X_test
    )

    pred_probs = booster.predict(
        dtest
    )

    y_pred = np.argmax(
        pred_probs,
        axis=1
    )

    all_y_true.append(
        y_test
    )

    all_y_pred.append(
        y_pred
    )

    del df
    del X_test
    del dtest

# =====================================================
# MERGE RESULTS ONLY
# =====================================================

y_true = np.concatenate(
    all_y_true
)

y_pred = np.concatenate(
    all_y_pred
)

# =====================================================
# METRICS
# =====================================================

print("\n========== FINAL RESULTS ==========")

acc = accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print(
    f"Accuracy : {acc:.4f}"
)

print(
    f"Macro F1 : {macro_f1:.4f}"
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print(
    f"\nConfusion Matrix Shape: "
    f"{cm.shape}"
)



Evaluating W-2022-44_features_test.parquet

Evaluating W-2022-45_features_test.parquet

Evaluating W-2022-46_features_test.parquet

Evaluating W-2022-47_features_test.parquet

========== FINAL RESULTS ==========
Accuracy : 0.6997
Macro F1 : 0.1385

Confusion Matrix Shape: (60, 60)
